<a href="https://colab.research.google.com/github/tanushreekaranth-cell/career-rag-project/blob/main/Copy_of_CARREER_RAG_ipynbCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu flask pyngrok
print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 41.7 MB/s eta 0:00:00
✅ All packages installed!


In [ ]:
from google.colab import files
import os

print("📂 Select your 4 PDF files")

uploaded = files.upload()

pdf_files = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):
        pdf_files.append(filename)

print("\n" + "=" * 50)
print("PDFs uploaded:", len(pdf_files))
print("=" * 50)

for filename in pdf_files:
    print("📄", filename)

if len(pdf_files) == 0:
    print(" No PDF files uploaded!")
else:
    print("\n PDFs are ready!")

📂 Select your 4 PDF files


Saving resume_guide_2020.pdf to resume_guide_2020.pdf
Saving Common-Job-Interview-Questions-and-Answers.pdf to Common-Job-Interview-Questions-and-Answers.pdf
Saving NITI_Internship_Guidelines.pdf to NITI_Internship_Guidelines.pdf

PDFs uploaded: 3
📄 resume_guide_2020.pdf
📄 Common-Job-Interview-Questions-and-Answers.pdf
📄 NITI_Internship_Guidelines.pdf

 PDFs are ready!


In [ ]:
from pypdf import PdfReader

documents = []

print("📖 Reading PDF files...\n")

for filename in pdf_files:

    try:

        reader = PdfReader(filename)

        text = ""

        for page in reader.pages:

            try:

                page_text = page.extract_text()

                if page_text is not None:
                    text += str(page_text) + "\n"

            except Exception:
                continue


        if text.strip():

            documents.append({
                "filename": filename,
                "text": text
            })

            print("✅", filename)
            print("   Characters:", len(text))

        else:

            print("⚠️", filename, "- no readable text")

    except Exception as error:

        print(" Error reading", filename)
        print(error)


print("\n==============================")
print("Readable PDFs:", len(documents))
print("==============================")

if len(documents) == 0:

    raise Exception(
        " No readable PDF text found."
    )

📖 Reading PDF files...

✅ resume_guide_2020.pdf
   Characters: 28829
✅ Common-Job-Interview-Questions-and-Answers.pdf
   Characters: 21254
✅ NITI_Internship_Guidelines.pdf
   Characters: 4051

Readable PDFs: 3


In [ ]:
CHUNK_SIZE = 500

chunks = []
sources = []

for document in documents:

    filename = document["filename"]
    text = document["text"]

    for start in range(
        0,
        len(text),
        CHUNK_SIZE
    ):

        chunk = text[
            start:start + CHUNK_SIZE
        ]

        if chunk.strip():

            chunks.append(chunk)
            sources.append(filename)


print(" Total chunks:", len(chunks))

if len(chunks) == 0:

    raise Exception(
        " No chunks were created."
    )

print(" Text chunking completed!")

 Total chunks: 110
 Text chunking completed!


In [ ]:
from sentence_transformers import SentenceTransformer

print(" Loading AI model...")

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✅ AI model loaded!")

 Loading AI model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ AI model loaded!


In [ ]:
# 🔹 Create embeddings + FAISS database

from sentence_transformers import SentenceTransformer
import faiss

# Load AI embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings from PDF chunks
embeddings = model.encode(
    chunks,
    convert_to_numpy=True
)

print("✅ Embeddings created!")
print("📊 Embedding shape:", embeddings.shape)

# Create FAISS database
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    embeddings.astype("float32")
)

print("✅ FAISS database created!")
print("📊 Stored vectors:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embeddings created!
📊 Embedding shape: (110, 384)
✅ FAISS database created!
📊 Stored vectors: 110


In [ ]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    embeddings.astype("float32")
)

print("✅ FAISS database created!")
print("📊 Stored vectors:", index.ntotal)

✅ FAISS database created!
📊 Stored vectors: 110


In [ ]:
TOP_K = 3

def search_careerrag(question):

    if question is None:

        return []


    question = str(question).strip()


    if question == "":

        return []


    question_embedding = model.encode(
        [question],
        convert_to_numpy=True
    )


    number_of_results = min(
        TOP_K,
        len(chunks)
    )


    distances, indices = index.search(
        question_embedding.astype("float32"),
        number_of_results
    )


    results = []


    for number, position in enumerate(
        indices[0],
        start=1
    ):

        results.append({
            "number": number,
            "text": chunks[position],
            "source": sources[position],
            "distance": float(
                distances[0][number - 1]
            )
        })


    return results


print("✅ Search system ready!")

✅ Search system ready!


In [ ]:
test_question = "What skills are required for placement?"

results = search_careerrag(
    test_question
)

print("\nCareerRAG Test Results")
print("=" * 60)

for result in results:

    print("\n Result", result["number"])

    print(result["text"])

    print("\n Source:", result["source"])

    print("-" * 60)


CareerRAG Test Results

 Result 1
VITIES
• Princeternship/shadowing, professional organizations or other activities aligned with career path
• Social clubs, sports teams, performance groups, etc., not listed in Leadership Roles section
SKILLS
Languages: Multilingual abilities (e.g., Fluent in Spanish) or computer programming (e.g., Proficient in C++)  
Certifications: Examples: CPR, Wildlife First Responder, Gold Award/Eagle Scout, technical training
Technology: Condense/expand list as needed based on your particular skills and 

 Source: resume_guide_2020.pdf
------------------------------------------------------------

 Result 2
lity
SKILLS
Languages: Multilingual abilities (Fluent in Spanish), separate from programming (Proficient in C++)  
Certifications: Examples: CPR, Wildlife First Responder, Gold Award/Eagle Scout, technical training
Technology: Software applications, hardware, and other tools relevant to your field(s) of interest
Additional Subcategories: Examples: Social Med

In [ ]:
import os

os.makedirs(
    "CareerRAG/templates",
    exist_ok=True
)

os.makedirs(
    "CareerRAG/static",
    exist_ok=True
)

os.makedirs(
    "CareerRAG/Documents",
    exist_ok=True
)

print(" Project folders created!")

 Project folders created!


In [ ]:
import shutil

for filename in pdf_files:

    shutil.copy(
        filename,
        os.path.join(
            "CareerRAG",
            "Documents",
            filename
        )
    )

print(" PDFs copied!")

for filename in os.listdir(
    "CareerRAG/Documents"
):

    print("📄", filename)

 PDFs copied!
📄 Common-Job-Interview-Questions-and-Answers.pdf
📄 resume_guide_2020.pdf
📄 NITI_Internship_Guidelines.pdf


In [ ]:
app_code = r'''
import os

from flask import (
    Flask,
    render_template,
    request,
    jsonify
)

from pypdf import PdfReader

from sentence_transformers import (
    SentenceTransformer
)

import faiss


# ============================================================
# FLASK APP
# ============================================================

app = Flask(__name__)


# ============================================================
# SETTINGS
# ============================================================

DOCUMENT_FOLDER = "Documents"

CHUNK_SIZE = 500

TOP_K = 3


# ============================================================
# PDF PROCESSING
# ============================================================

chunks = []

sources = []


if os.path.exists(DOCUMENT_FOLDER):

    pdf_files = [
        file
        for file in os.listdir(DOCUMENT_FOLDER)
        if file.lower().endswith(".pdf")
    ]

else:

    pdf_files = []


for filename in pdf_files:

    filepath = os.path.join(
        DOCUMENT_FOLDER,
        filename
    )

    try:

        reader = PdfReader(filepath)

        text = ""


        for page in reader.pages:

            try:

                extracted = page.extract_text()

                if extracted is not None:

                    text += str(extracted) + "\n"

            except Exception:

                continue


        for start in range(
            0,
            len(text),
            CHUNK_SIZE
        ):

            chunk = text[
                start:start + CHUNK_SIZE
            ]


            if chunk.strip():

                chunks.append(chunk)

                sources.append(filename)


    except Exception:

        continue


# ============================================================
# AI MODEL
# ============================================================

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# FAISS
# ============================================================

if len(chunks) > 0:

    embeddings = model.encode(
        chunks,
        convert_to_numpy=True
    )


    dimension = embeddings.shape[1]


    index = faiss.IndexFlatL2(
        dimension
    )


    index.add(
        embeddings.astype("float32")
    )

else:

    index = None


# ============================================================
# HOME PAGE
# ============================================================

@app.route("/")
def home():

    return render_template(
        "index.html"
    )


# ============================================================
# SEARCH API
# ============================================================

@app.route(
    "/search",
    methods=["POST"]
)
def search():

    data = request.get_json(
        silent=True
    )


    if data is None:

        return jsonify({
            "html":
            "<div class='error'>Invalid request.</div>"
        })


    question = data.get(
        "question",
        ""
    )


    if question is None:

        question = ""


    question = str(
        question
    ).strip()


    if question == "":

        return jsonify({
            "html":
            "<div class='error'>Please enter a question.</div>"
        })


    if index is None or len(chunks) == 0:

        return jsonify({
            "html":
            "<div class='error'>No document data available.</div>"
        })


    question_embedding = model.encode(
        [question],
        convert_to_numpy=True
    )


    number_of_results = min(
        TOP_K,
        len(chunks)
    )


    distances, indices = index.search(
        question_embedding.astype(
            "float32"
        ),
        number_of_results
    )


    html = ""


    for number, position in enumerate(
        indices[0],
        start=1
    ):

        text = chunks[position]

        source = sources[position]


        safe_text = (
            text
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace("\n", "<br>")
        )


        html += f"""

        <div class="result-card">

            <div class="result-number">
                RESULT {number}
            </div>

            <h3>
                📌 Relevant Information
            </h3>

            <p>
                {safe_text}
            </p>

            <div class="source">
                📄 Source: {source}
            </div>

        </div>

        """


    return jsonify({
        "html": html
    })


# ============================================================
# RUN APP
# ============================================================

if __name__ == "__main__":

    app.run(
        host="0.0.0.0",
        port=5000
    )
'''


with open(
    "CareerRAG/app.py",
    "w",
    encoding="utf-8"
) as file:

    file.write(app_code)


print("✅ app.py created!")

✅ app.py created!


In [ ]:
html_code = r'''
<!DOCTYPE html>

<html lang="en">

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <title>CareerRAG</title>

    <link rel="stylesheet"
          href="{{ url_for('static', filename='style.css') }}">

</head>


<body>


<header class="navbar">

    <div class="logo">
        🎓 CareerRAG
    </div>

    <div class="nav-text">
        Career & Placement Assistant
    </div>

</header>


<main>


<section class="hero">

    <div class="badge">
        AI POWERED • SEMANTIC SEARCH
    </div>


    <h1>

        Your Career Questions.
        <br>

        <span>Answered from Your Documents.</span>

    </h1>


    <p class="subtitle">

        Search your placement, internship and
        career documents using AI-powered
        semantic search.

    </p>


    <div class="search-container">

        <input
            id="question"
            type="text"
            placeholder="Ask about placements, internships, skills..."
        >

        <button
            id="searchButton"
            onclick="searchCareer()">

            🔍 Search

        </button>

    </div>


    <div class="suggestions">

        <button onclick="setQuestion(
        'What skills are required for placement?')">

            💼 Placement Skills

        </button>


        <button onclick="setQuestion(
        'What are the internship requirements?')">

            🎯 Internship

        </button>


        <button onclick="setQuestion(
        'What is the placement eligibility criteria?')">

            📋 Eligibility

        </button>


    </div>

</section>


<section
    id="loading"
    class="loading"
    style="display:none;">

    <div class="loader"></div>

    <span>
        Searching your documents...
    </span>

</section>


<section
    id="results"
    class="results">

</section>


<section class="features">


    <div class="feature-card">

        <div class="feature-icon">

        </div>

        <h3>
            Document Based
        </h3>

        <p>
            Answers are retrieved from
            your uploaded documents.
        </p>

    </div>


    <div class="feature-card">

        <div class="feature-icon">
            🧠
        </div>

        <h3>
            Semantic Search
        </h3>

        <p>
            Understands the meaning
            behind your question.
        </p>

    </div>


    <div class="feature-card">

        <div class="feature-icon">
            ⚡
        </div>

        <h3>
            Fast Retrieval
        </h3>

        <p>
            FAISS finds relevant
            information quickly.
        </p>

    </div>


</section>


</main>


<footer>

    <strong>
        CareerRAG
    </strong>

    <span>
        • AI Career & Placement Assistant
    </span>

</footer>


<script>


function setQuestion(text) {

    document.getElementById(
        "question"
    ).value = text;

}


async function searchCareer() {


    const question =
        document.getElementById(
            "question"
        ).value;


    if (!question.trim()) {

        alert(
            "Please enter a question."
        );

        return;

    }


    const loading =
        document.getElementById(
            "loading"
        );


    const results =
        document.getElementById(
            "results"
        );


    const button =
        document.getElementById(
            "searchButton"
        );


    loading.style.display =
        "flex";


    results.innerHTML = "";


    button.disabled = true;


    try {


        const response =
            await fetch(
                "/search",
                {

                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        question: question
                    })

                }
            );


        const data =
            await response.json();


        results.innerHTML =
            data.html;


        results.scrollIntoView({
            behavior: "smooth"
        });


    }


    catch(error) {


        results.innerHTML = `

            <div class="error">

                ❌ Unable to search.
                Please try again.

            </div>

        `;

    }


    loading.style.display =
        "none";


    button.disabled =
        false;

}


document
    .getElementById("question")
    .addEventListener(
        "keypress",
        function(event) {

            if (
                event.key === "Enter"
            ) {

                searchCareer();

            }

        }
    );


</script>


</body>

</html>
'''


with open(
    "CareerRAG/templates/index.html",
    "w",
    encoding="utf-8"
) as file:

    file.write(html_code)


print(" index.html created!")

 index.html created!


In [ ]:
css_code = r'''
* {

    margin: 0;

    padding: 0;

    box-sizing: border-box;

}


body {

    font-family:
        Arial,
        Helvetica,
        sans-serif;

    background: #f5f7fb;

    color: #202124;

    min-height: 100vh;

}


/* =========================
   NAVBAR
========================= */

.navbar {

    height: 75px;

    background: white;

    border-bottom:
        1px solid #e5e7eb;

    display: flex;

    align-items: center;

    justify-content: space-between;

    padding: 0 8%;

}


.logo {

    font-size: 25px;

    font-weight: 700;

}


.nav-text {

    color: #6b7280;

    font-size: 14px;

}


/* =========================
   MAIN
========================= */

main {

    max-width: 1100px;

    margin: auto;

    padding: 70px 20px;

}


/* =========================
   HERO
========================= */

.hero {

    text-align: center;

}


.badge {

    display: inline-block;

    padding: 8px 15px;

    border-radius: 30px;

    background: white;

    border:
        1px solid #e5e7eb;

    font-size: 12px;

    font-weight: 700;

    letter-spacing: 1px;

    margin-bottom: 25px;

}


.hero h1 {

    font-size: 48px;

    line-height: 1.15;

    margin-bottom: 25px;

}


.hero h1 span {

    font-weight: 700;

}


.subtitle {

    max-width: 680px;

    margin: auto;

    color: #6b7280;

    font-size: 18px;

    line-height: 1.6;

    margin-bottom: 35px;

}


/* =========================
   SEARCH
========================= */

.search-container {

    max-width: 800px;

    margin: auto;

    padding: 8px;

    background: white;

    border:
        1px solid #e1e4e8;

    border-radius: 16px;

    display: flex;

    box-shadow:
        0 10px 30px rgba(
            0,
            0,
            0,
            0.07
        );

}


.search-container input {

    flex: 1;

    border: none;

    outline: none;

    padding: 17px;

    font-size: 16px;

    background: transparent;

}


.search-container button {

    border: none;

    border-radius: 11px;

    padding: 0 28px;

    font-size: 15px;

    font-weight: 600;

    cursor: pointer;

}


.search-container button:disabled {

    opacity: 0.6;

    cursor: not-allowed;

}


/* =========================
   SUGGESTIONS
========================= */

.suggestions {

    margin-top: 18px;

}


.suggestions button {

    background: white;

    border:
        1px solid #e1e4e8;

    padding: 10px 15px;

    border-radius: 25px;

    margin: 5px;

    cursor: pointer;

    font-size: 13px;

}


.suggestions button:hover {

    transform: translateY(-2px);

}


/* =========================
   LOADING
========================= */

.loading {

    justify-content: center;

    align-items: center;

    gap: 12px;

    margin-top: 40px;

    color: #6b7280;

}


.loader {

    width: 22px;

    height: 22px;

    border:
        3px solid #ddd;

    border-top:
        3px solid #555;

    border-radius: 50%;

    animation:
        spin 1s linear infinite;

}


@keyframes spin {

    100% {

        transform:
            rotate(360deg);

    }

}


/* =========================
   RESULTS
========================= */

.results {

    max-width: 850px;

    margin: 50px auto 0;

}


.result-card {

    background: white;

    border:
        1px solid #e5e7eb;

    border-radius: 16px;

    padding: 25px;

    margin-bottom: 20px;

    box-shadow:
        0 6px 20px rgba(
            0,
            0,
            0,
            0.05
        );

}


.result-number {

    font-size: 11px;

    font-weight: 700;

    letter-spacing: 1px;

    margin-bottom: 8px;

    color: #6b7280;

}


.result-card h3 {

    margin-bottom: 15px;

    font-size: 19px;

}


.result-card p {

    line-height: 1.7;

    font-size: 15px;

}


.source {

    margin-top: 18px;

    padding-top: 15px;

    border-top:
        1px solid #eee;

    font-size: 13px;

    color: #6b7280;

}


/* =========================
   FEATURES
========================= */

.features {

    display: grid;

    grid-template-columns:
        repeat(
            3,
            1fr
        );

    gap: 20px;

    margin-top: 70px;

}


.feature-card {

    background: white;

    padding: 25px;

    border:
        1px solid #e5e7eb;

    border-radius: 15px;

}


.feature-icon {

    font-size: 28px;

    margin-bottom: 15px;

}


.feature-card h3 {

    margin-bottom: 10px;

}


.feature-card p {

    color: #6b7280;

    line-height: 1.5;

    font-size: 14px;

}


/* =========================
   ERROR
========================= */

.error {

    background: #fff1f2;

    border:
        1px solid #fecdd3;

    padding: 20px;

    border-radius: 12px;

    margin-bottom: 20px;

}


/* =========================
   FOOTER
========================= */

footer {

    text-align: center;

    padding: 30px;

    color: #6b7280;

    border-top:
        1px solid #e5e7eb;

    background: white;

}


footer strong {

    color: #202124;

}


/* =========================
   MOBILE
========================= */

@media (
    max-width: 700px
) {


    .navbar {

        padding: 0 20px;

    }


    .nav-text {

        display: none;

    }


    main {

        padding:
            45px 15px;

    }


    .hero h1 {

        font-size: 35px;

    }


    .subtitle {

        font-size: 16px;

    }


    .search-container {

        flex-direction: column;

        gap: 5px;

    }


    .search-container button {

        padding: 15px;

    }


    .features {

        grid-template-columns:
            1fr;

    }

}
'''


with open(
    "CareerRAG/static/style.css",
    "w",
    encoding="utf-8"
) as file:

    file.write(css_code)


print(" style.css created!")

 style.css created!


In [ ]:
requirements = """Flask
pypdf
sentence-transformers
faiss-cpu
gunicorn
"""

with open(
    "CareerRAG/requirements.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(requirements)

print("✅ requirements.txt created!")

✅ requirements.txt created!


In [ ]:
import os

print("📁 CareerRAG PROJECT")
print("=" * 50)

for root, dirs, files_list in os.walk(
    "CareerRAG"
):

    level = root.replace(
        "CareerRAG",
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        indent +
        "📂 " +
        os.path.basename(root)
    )

    for file in files_list:

        print(
            indent +
            "    📄 " +
            file
        )

📁 CareerRAG PROJECT
📂 CareerRAG
    📄 app.py
    📄 requirements.txt
    📂 static
        📄 style.css
    📂 templates
        📄 index.html
    📂 Documents
        📄 Common-Job-Interview-Questions-and-Answers.pdf
        📄 resume_guide_2020.pdf
        📄 NITI_Internship_Guidelines.pdf


In [ ]:
import subprocess
import time

process = subprocess.Popen(
    [
        "python",
        "app.py"
    ],
    cwd="CareerRAG"
)

time.sleep(5)

print(" CareerRAG server started!")
print("Port: 5000")

 CareerRAG server started!
Port: 5000


In [ ]:
# 🚀 CareerRAG Web Interface - Complete

import gradio as gr
import numpy as np

def search(question):

    if not question or not question.strip():
        return "⚠️ Please enter a question."

    # Convert question into embedding
    q = model.encode(
        [question],
        convert_to_numpy=True
    )

    # Search FAISS
    distances, ids = index.search(
        q.astype("float32"),
        min(3, len(chunks))
    )

    result = "## 🤖 CareerRAG Results\n\n"

    for n, i in enumerate(ids[0], 1):

        result += f"### 📌 Result {n}\n\n"
        result += chunks[i]
        result += f"\n\n📄 **Source:** {sources[i]}\n\n"
        result += "---\n\n"

    return result


# 🌐 Website
demo = gr.Interface(
    fn=search,

    inputs=gr.Textbox(
        label="💬 Ask CareerRAG",
        placeholder="Ask about placements, internships, skills...",
        lines=2
    ),

    outputs=gr.Markdown(),

    title="🎓 CareerRAG",
    description="AI Career & Placement Document Assistant",

    examples=[
        "What skills are required for placement?",
        "What are the internship requirements?",
        "What is the placement eligibility criteria?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6a29591f43a796c63e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
